## Exercise 3.1 Work with SAP HANA Cloud Vector Engine from LangChain

In this exercise we will use SAP HANA Cloud as a vector store in LangChain. We will use the python packages `sap-ai-sdk-gen`, `langchain`, and `langchain-hana`.

### Connect to SAP HANA Cloud

First, we need to connect to our SAP HANA Cloud instance (hana-db) running on SAP BTP. Use your database credential

In [20]:
import os
from dotenv import load_dotenv
from hdbcli import dbapi

load_dotenv()

connection = dbapi.connect(
    user="DA263_000", # (replace 000 with your student number, e.g. 001)
    password=os.environ.get("HANA_DB_PASSWORD"),
    address=os.environ.get("HANA_DB_ADDRESS"),
    port=os.environ.get("HANA_DB_PORT"),
    # sslValidateCertificate = False
)

print('HANA Cloud connected:', connection.isconnected())

HANA Cloud connected: True


Let's see how HAN Cloud is used as a `vector store` in LangChain. A `vector store` is nothing but a table with a specific layout which contains a column with text embedding vector. A `vector store` is used for semantic search and provides context for an LLM workflow. We will use the in-database embedding model `SAP_GXY.20250407` to create the text embeddings.

In [ ]:
from langchain_hana import HanaDB
from langchain_hana import HanaInternalEmbeddings

embeddings = HanaInternalEmbeddings(internal_embedding_model_id="SAP_GXY.20250407")

# creating a vector store, i.e. a HanaDB instance, creates a simple table in HANA Cloud with 3 columns: VEC_TEXT, VEC_META, VEC_VECTOR
# If the table already exists, it is checked and used.
db = HanaDB(
    connection=connection, 
    embedding=embeddings, 
    table_name="LC_HANA_VECTOR_STORE"
)

We can now use the vector store to store documents. The text embeddings are created automatically in HANA Cloud.

In [8]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="SAP TechEd is SAP's premier annual technical conference designed for developers, engineers, architects, and IT professionals.",
        metadata={"source": "SAP", "confidence": 1.0},
    ),
    Document(
        page_content="SAP Sapphire is the flagship annual business technology conference hosted by SAP, one of the world's largest enterprise software companies.",
        metadata={"source": "SAP", "confidence": 1.0},
    ),
    Document(
            page_content="Sapperlot is a traditional German and regional interjection used to express surprise, amazement, annoyance, or mild indignation.",
            metadata={"source": "Internet", "confidence": 0.8},
        ),
]

db.add_documents(docs)

[]

Let's run a semantic search on the vector store.

In [15]:
# scores are cosine similarity in [0, 1]; higher = more similar
docs = db.similarity_search_with_score("conference for developers", k=3)

for doc, score in docs:
    print("-" * 80)
    print(f"Score: {score}")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

--------------------------------------------------------------------------------
Score: 0.45682889493774803
Content: Stores the Formula 1 Constructors' Championship standings after each race. Each row represents the cumulative standing of a constructor (team) at a specific point in the season, including total points, position, and number of wins.
Metadata: {'OBJECT_OID': 166708, 'SCHEMA_NAME': 'ERGAST_2025', 'OBJECT_NAME': 'CONSTRUCTORSTANDINGS', 'OBJECT_TYPE': 'TABLE'}
--------------------------------------------------------------------------------
Score: 0.451379154670761
Content: Lookup table containing the possible finishing status codes for drivers in a race. Each row maps a unique status identifier to a descriptive label such as 'Finished', 'Retired', 'Accident', 'Engine', etc.
Metadata: {'OBJECT_OID': 166848, 'SCHEMA_NAME': 'ERGAST_2025', 'OBJECT_NAME': 'STATUS', 'OBJECT_TYPE': 'TABLE'}
--------------------------------------------------------------------------------
Score: 0.449

In [16]:
# we can also use a filter expression on the metadata, e.g. "confidence" > 0.9
docs = db.similarity_search_with_score("conference for developers", k=3, filter={"confidence": {'$gt': 0.9}})

for doc, score in docs:
    print("-" * 80)
    print(f"Score: {score}")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

--------------------------------------------------------------------------------
0.635400259230992
SAP TechEd is SAP's premier annual technical conference designed for developers, engineers, architects, and IT professionals.
{'source': 'SAP', 'confidence': 1.0}
--------------------------------------------------------------------------------
0.5555538488141406
SAP Sapphire is the flagship annual business technology conference hosted by SAP, one of the world's largest enterprise software companies.
{'source': 'SAP', 'confidence': 1.0}


a vector store can be used as a retriever

In [18]:
retriever = db.as_retriever()
docs = retriever.invoke("developer conference", filter={"confidence": {'$gt': 0.9}})
for doc in docs:
    print(f"Score: {score}")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

--------------------------------------------------------------------------------
SAP TechEd is SAP's premier annual technical conference designed for developers, engineers, architects, and IT professionals.
{'source': 'SAP', 'confidence': 1.0}
--------------------------------------------------------------------------------
SAP Sapphire is the flagship annual business technology conference hosted by SAP, one of the world's largest enterprise software companies.
{'source': 'SAP', 'confidence': 1.0}


In exercise 1 we have created an `OBJECT_INDEX` table to store metadata of database objects, i.e. tables in the ERGAST_2025 schema.
To use this custom table as a LangChain vector store, we need to describe it's schema. The `specific_metadata_columns` are used for fast filtering. Filter predicates like SCHEMA_NAME = 'ERGAST_2025' are handled much faster on these dedicated table columns compared to the same data stored as JSON in the `LC_METADATA` column.

In [21]:
from langchain_hana import HanaDB
from langchain_hana import HanaInternalEmbeddings

embeddings = HanaInternalEmbeddings(internal_embedding_model_id="SAP_GXY.20250407")

db = HanaDB(
    connection=connection,
    embedding=embeddings,
    table_name="OBJECT_INDEX",
    content_column="DESCRIPTION",
    metadata_column="LC_METADATA",
    vector_column="DESCRIPTION_EMBEDDING",
    specific_metadata_columns=["OBJECT_OID", "SCHEMA_NAME", "OBJECT_NAME", "OBJECT_TYPE"],
)

Now let's search for a database object in our index which provides information about formula 1 pilots. We expect to find the `DRIVERS` table.

In [22]:
query = "f1 pilots"
advanced_filter = {"SCHEMA_NAME": "ERGAST_2025"}
docs = db.similarity_search_with_score(query, k=3, filter=advanced_filter)
for doc, score in docs:
    print("-" * 80)
    print(f"Score: {score}")
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}")

--------------------------------------------------------------------------------
Score: 0.6247333665445782
Metadata: {'OBJECT_OID': 166719, 'SCHEMA_NAME': 'ERGAST_2025', 'OBJECT_NAME': 'DRIVERS', 'OBJECT_TYPE': 'TABLE'}
Content: Contains information about Formula 1 drivers, including their name, date of birth, nationality, driver code, and permanent race number. Each row represents a unique driver.
--------------------------------------------------------------------------------
Score: 0.5888581136077121
Metadata: {'OBJECT_OID': 166740, 'SCHEMA_NAME': 'ERGAST_2025', 'OBJECT_NAME': 'LAPTIMES', 'OBJECT_TYPE': 'TABLE'}
Content: Records the lap time for every driver on every lap of each Formula 1 race. Each row represents a single lap by a driver, capturing the lap number, time, position on track at the end of the lap, and duration in milliseconds.
--------------------------------------------------------------------------------
Score: 0.5826902400551153
Metadata: {'OBJECT_OID': 166753, 'SCH

Let's use an LLM to generate the queries for the vector store. The LLM is running on AI Core and is exposed via gen-ai-hub sdk to LangChain.

In [5]:
from gen_ai_hub.proxy.langchain import ChatOpenAI
from gen_ai_hub.proxy import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')

llm = ChatOpenAI(proxy_model_name='gpt-4.1', proxy_client=proxy_client)

print("LLM model:", llm.model_name)

LLM model: gpt-4.1


creating a self query retriever

In [9]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_hana import HanaTranslator

metadata_field_info = [
    AttributeInfo(
        name="OBJECT_OID",
        description="The ID of the database object.",
        type="integer",
    ),
    AttributeInfo(
        name="SCHEMA_NAME",
        description="The schema name where the object resides.",
        type="string",
    ),
    AttributeInfo(
        name="OBJECT_NAME",
        description="The name of the database object.",
        type="string",
    ),
    AttributeInfo(
        name="OBJECT_TYPE",
        description="The type of the database object, e.g. TABLE, VIEW, FUNCTION.",
        type="string",
    ),
]

document_content_description = "A catalog of database objects."

hana_translator = HanaTranslator()

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=db,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    structured_query_translator=hana_translator,
)

In [17]:
query_prompt = "which views are in the DA263_000 schema"
query_prompt = "tables in the ERGAST_2025 schema f1 pilots"

docs = retriever.invoke(input=query_prompt)
for doc in docs:
    print("-" * 80)
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}")

--------------------------------------------------------------------------------
Metadata: {'OBJECT_OID': 166719, 'SCHEMA_NAME': 'ERGAST_2025', 'OBJECT_NAME': 'DRIVERS', 'OBJECT_TYPE': 'TABLE'}
Content: Contains information about Formula 1 drivers, including their name, date of birth, nationality, driver code, and permanent race number. Each row represents a unique driver.
--------------------------------------------------------------------------------
Metadata: {'OBJECT_OID': 166740, 'SCHEMA_NAME': 'ERGAST_2025', 'OBJECT_NAME': 'LAPTIMES', 'OBJECT_TYPE': 'TABLE'}
Content: Records the lap time for every driver on every lap of each Formula 1 race. Each row represents a single lap by a driver, capturing the lap number, time, position on track at the end of the lap, and duration in milliseconds.
--------------------------------------------------------------------------------
Metadata: {'OBJECT_OID': 166753, 'SCHEMA_NAME': 'ERGAST_2025', 'OBJECT_NAME': 'PITSTOPS', 'OBJECT_TYPE': 'TABLE'}
C

In [19]:
from langchain_classic.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)

prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
)
output_parser = StructuredQueryOutputParser.from_components()
query_constructor = prompt | llm | output_parser

sq = query_constructor.invoke(input=query_prompt)

print("Structured query:\n", sq)

print("\nTranslated for HANA vector store:\n", hana_translator.visit_structured_query(sq))

Structured query:
 query='f1 pilots' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='SCHEMA_NAME', value='ERGAST_2025'), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='OBJECT_TYPE', value='TABLE')]) limit=None

Translated for HANA vector store:
 ('f1 pilots', {'filter': {'$and': [{'SCHEMA_NAME': {'$eq': 'ERGAST_2025'}}, {'OBJECT_TYPE': {'$eq': 'TABLE'}}]}})
